# 第 3 章 02：观测、策略与价值函数

这一课回答三个相连的问题：**智能体真的看得到完整状态吗？策略到底输出什么？怎样评价一个局面或一个动作？** 读完后应能区分状态与观测、状态价值与动作价值，并能用超级玛丽理解最优动作价值。


## 1. 状态不必包含所有信息，但必须足够决策

理想状态 $S_t$ 要满足马尔科夫性质：给定当前状态和动作，下一状态与奖励的分布不再需要更早历史。它不要求记录环境里一切无关信息；只要求保留会影响决策后果的信息。

例如下棋时，棋盘局面通常足够，不需要知道棋子是怎样走到这里的。锁门迷宫中，状态若只写位置就不够；还应写上是否有钥匙。


## 2. 现实中常只有观测

很多任务里，智能体看不到真实状态 $S_t$，只能收到观测 $O_t$。单个观测通常不包含做决策所需的全部信息，这类问题称为部分可观测问题。

以《红色警戒》为例，当前屏幕画面只是局部观测：战争迷雾下的敌军、屏幕外的建筑和敌人的生产队列都不可见。滑动屏幕、小地图和侦察能获得更多观测；把看到的地图、单位与历史行动合并后，玩家在脑中形成更好的内部状态：

$$
z_t=f(z_{t-1},O_t,A_{t-1}).
$$

即使如此，未侦察区域仍有不确定性。此时 $z_t$ 是状态估计，或更严格地说是对真实状态的信念，而不是完整环境状态本身。


## 3. 策略：给定局面后如何选动作

策略 $\pi(a\mid s)$ 表示：已知状态为 $s$ 时，选择动作 $a$ 的条件概率分布。它不是直接说“哪个动作一定发生”，而是给每个候选动作分配概率。

书中的超级玛丽例子把当前屏幕画面当作状态 $s$，动作空间为 $\{\text{左},\text{右},\text{上}\}$。例如：

$$
\\pi(\text{左}\mid s)=0.2,\qquad
\\pi(\text{右}\mid s)=0.1,\qquad
\\pi(\text{上}\mid s)=0.7.
$$

随后按这三个概率随机抽取一个动作。因为动作是离散的，严格说这里是条件概率质量函数（PMF），三项之和为 $1$；书中“条件概率密度函数”的说法是更宽松的总称。若动作是连续的，例如方向角或推力，策略才常用条件概率密度来描述。


## 4. 动作价值：现在先做这一步，长远值不值

从时刻 $t$ 开始的折扣累计回报记为 $G_t$。它不是当前一步的奖励，而是之后所有奖励的加权和：

$$
G_t=R_{t+1}+\gamma R_{t+2}+\gamma^2R_{t+3}+\cdots.
$$

动作价值函数问的是：在状态 $s$，当前这一步指定做动作 $a$，从下一步起持续按策略 $\pi$ 行动，平均能得到多少累计回报？

$$
Q_\pi(s,a)=\mathbb{E}[G_t\mid S_t=s,A_t=a].
$$

这里固定的是**当前这一动作**，不是以后每一步都重复 $a$。例如 $Q_\pi(s,\text{上跳})$ 是“现在先上跳，之后按 $\pi$ 玩”的长期平均结果。


## 5. 那一长串期望推导在说什么

书中把 $Q_\pi(s_t,a_t)$ 展开为对未来状态、未来动作的多重积分和概率连乘。不要把它当成新的概念；它只是把下面这句话完整写出来：

> 现在处于 $s_t$，先做 $a_t$。列出之后所有可能的完整游戏剧情；每条剧情用“发生概率 $\\times$ 该剧情的累计回报”，最后全部相加。

一条未来剧情包含 $S_{t+1},A_{t+1},S_{t+2},A_{t+2},\ldots$。其中环境转移概率 $P(s_{k+1}\mid s_k,a_k)$ 决定画面如何变化，策略概率 $\pi(a_{k+1}\mid s_{k+1})$ 决定之后怎样选动作；把它们连乘，就得到整条剧情发生的概率。连续空间用积分表示“遍历所有可能”，离散的超级玛丽动作可直观地理解成求和。

真正做强化学习时，通常不手工枚举这些剧情；可以像上一章那样采样多条轨迹，用实际回报的平均近似这个期望。


## 6. 状态价值：这个局面整体前景如何

状态价值函数不指定当前动作：从状态 $s$ 出发，连当前动作也按策略 $\pi$ 的概率选择，之后持续按该策略行动，平均回报是多少？

$$
V_\pi(s)=\mathbb{E}[G_t\mid S_t=s].
$$

因此，状态价值是各个动作价值按策略概率做的加权平均：

$$
V_\pi(s)=\sum_{a\in\mathcal{A}}\pi(a\mid s)Q_\pi(s,a).
$$

例如策略有 $0.2$ 概率向左、$0.1$ 概率向右、$0.7$ 概率向上，而三个动作价值分别为 $10$、$20$、$100$，则：

$$
V_\pi(s)=0.2\times10+0.1\times20+0.7\times100=74.
$$

所以 $V_\pi(s)$ 问“这个局面在当前玩法下怎么样”，$Q_\pi(s,a)$ 问“这个局面先做这个动作值不值”。


## 7. 最优动作价值：先试一个动作，之后都做到最好

最优动作价值函数为：

$$
Q_\star(s,a)=\max_\pi Q_\pi(s,a).
$$

它的含义是：当前先强制执行 $a$；从下一步开始，采用能让回报最大的最优策略。它仍会给每个当前候选动作打分，即使那个动作本身不是最佳选择。

一旦知道 $Q_\star$，最优策略就在每个状态选分数最高的当前动作：

$$
\pi_\star(s)=\underset{a}{\operatorname{argmax}}\ Q_\star(s,a).
$$

只有“当前也选最高分动作，并且之后每一步都这样选”，才是在始终使用最优策略。


## 8. 小结与自检

- 真实状态要包含预测决策后果所需的信息；观测只是智能体实际看见的信息。
- 策略 $\pi(a\mid s)$ 是给定状态后的动作条件分布。
- $Q_\pi(s,a)$ 固定当前动作；$V_\pi(s)$ 连当前动作也按策略随机选择。
- $Q_\star(s,a)$ 固定当前动作后，假设未来采取最优策略。

自检：$Q_\pi(s,a)$ 和 $Q_\star(s,a)$ 都固定了当前动作 $a$，二者唯一且关键的差别是什么？

答：$Q_\pi$ 的未来动作按指定策略 $\pi$ 选择；$Q_\star$ 的未来动作按最优策略选择，因此它是在所有可选未来策略中取最大期望回报。
